# Validation-only synonymous-family decoding diagnosis

Diagnose E/K/L/S conditional probabilities for the official pretrained baseline and formal v1 checkpoint, then compare translation-preserving greedy, temperature, and synonymous-family sampling. This notebook does not train and never reads the test split.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "CUDA GPU is required"
print(torch.cuda.get_device_name(0))

## Clone the private repository securely
Store a fine-grained read-only token as the Colab secret `GITHUB_TOKEN`. Authentication uses `GIT_ASKPASS`; the token is not printed or embedded in a command.

In [ ]:
import hashlib
import json
import os
import subprocess
import tempfile
from pathlib import Path
from google.colab import userdata

REPO_URL = "https://github.com/Yuano0o/codontransformer-nb.git"
PROJECT_DIR = Path("/content/codontransformer-project")
UPSTREAM_URL = "https://github.com/Adibvafa/CodonTransformer.git"
UPSTREAM_COMMIT = "4a447b01dab860feb81b647ff1ff88ad598517f4"
HF_MODEL_ID = "adibvafa/CodonTransformer"
HF_MODEL_REVISION = "9744dcc920d813066391fc828d7a590207f148e8"
github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab secret GITHUB_TOKEN is unavailable")
with tempfile.TemporaryDirectory(prefix="git-askpass-") as askpass_dir:
    askpass = Path(askpass_dir) / "askpass.sh"
    askpass.write_text(
        '#!/bin/sh\ncase "$1" in\n'
        '  *sername*) printf \'%s\\n\' \'x-access-token\' ;;\n'
        '  *assword*) printf \'%s\\n\' "$COLAB_GITHUB_TOKEN" ;;\n'
        '  *) exit 1 ;;\nesac\n', encoding="utf-8")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update({"GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0", "COLAB_GITHUB_TOKEN": github_token})
    if not (PROJECT_DIR / ".git").is_dir():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True, env=env)
    else:
        subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True, env=env)
    env.pop("COLAB_GITHUB_TOKEN", None)
github_token = None
os.chdir(PROJECT_DIR)
UPSTREAM_DIR = PROJECT_DIR / "upstream"
if not (UPSTREAM_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", UPSTREAM_URL, str(UPSTREAM_DIR)], check=True)
subprocess.run(["git", "-C", str(UPSTREAM_DIR), "checkout", UPSTREAM_COMMIT], check=True)

## Install dependencies and mount Google Drive

In [ ]:
subprocess.run([os.sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], check=True)
os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, (str(UPSTREAM_DIR), os.environ.get("PYTHONPATH", ""))))
if str(UPSTREAM_DIR) not in os.sys.path:
    os.sys.path.insert(0, str(UPSTREAM_DIR))
from google.colab import drive
drive.mount("/content/drive")

## Verify the frozen validation inputs

The configuration pins the 531-record validation hash, codon-reference hash, pretrained snapshot, v1 checkpoint identity, E/K/L/S families, fixed seed, and refined-v2 length boundaries.

In [ ]:
import yaml
CONFIG_PATH = PROJECT_DIR / "configs/diagnose_validation_decoding.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
assert CONFIG["dataset_role"] == "validation"
DRIVE_ROOT = Path("/content/drive/MyDrive/CodonTransformer")
VALIDATION_PATH = DRIVE_ROOT / CONFIG["paths"]["validation_dataset"]
REFERENCE_PATH = DRIVE_ROOT / CONFIG["paths"]["codon_reference"]
RUN_DIR = DRIVE_ROOT / CONFIG["paths"]["formal_run"]
CHECKPOINT = RUN_DIR / "checkpoints" / CONFIG["inputs"]["checkpoint_filename"]
OUTPUT_DIR = RUN_DIR / CONFIG["paths"]["output_directory"]

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

for label, path in (("validation", VALIDATION_PATH), ("reference", REFERENCE_PATH), ("checkpoint", CHECKPOINT)):
    assert path.is_file(), f"Missing {label}: {path}"
with VALIDATION_PATH.open(encoding="utf-8") as handle:
    assert sum(bool(line.strip()) for line in handle) == 531
assert sha256(VALIDATION_PATH) == CONFIG["inputs"]["validation_sha256"]
assert sha256(REFERENCE_PATH) == CONFIG["inputs"]["reference_sha256"]
assert CHECKPOINT.stat().st_size == CONFIG["inputs"]["checkpoint_size_bytes"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Persistent validation-only output:", OUTPUT_DIR)

## Download and verify the exact pretrained baseline

In [ ]:
MODEL_DIR = PROJECT_DIR / "models/pretrained"
subprocess.run([
    os.sys.executable, "scripts/download_pretrained.py",
    "--repo-id", HF_MODEL_ID, "--revision", HF_MODEL_REVISION,
    "--output-dir", str(MODEL_DIR),
], check=True)
assert sha256(MODEL_DIR / "model.safetensors") == CONFIG["inputs"]["pretrained_model_safetensors_sha256"]

## Run or resume probability diagnosis and decoding strategies

Each model writes one record cache to Drive every 10 newly processed records. Per-record sampling seeds are derived from the frozen seed, model, strategy, and idx, so interruption and resume do not change any output. All strategies hard-mask non-synonymous tokens.

In [ ]:
subprocess.run([
    os.sys.executable, "scripts/diagnose_validation_decoding.py",
    "--config", str(CONFIG_PATH),
    "--model-dir", str(MODEL_DIR),
    "--checkpoint", str(CHECKPOINT),
    "--validation-dataset", str(VALIDATION_PATH),
    "--reference-json", str(REFERENCE_PATH),
    "--output-dir", str(OUTPUT_DIR),
    "--force-analysis",
], check=True)

## Inspect the collapse diagnosis and saved artifacts

In [ ]:
SUMMARY_PATH = OUTPUT_DIR / "decoding_diagnostic_summary.json"
summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
assert summary["dataset_role"] == "validation"
assert summary["records"] == 531
assert summary["test_access_prohibited"] is True
print(json.dumps(summary["collapse_diagnosis"], indent=2))
print((OUTPUT_DIR / "decoding_diagnostic_report.md").read_text())
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR), path.stat().st_size)